In [ ]:
!pip install transformers torch scikit-learn


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader

SAMPLE SENTIMENT

In [ ]:
texts = [
    "I love this product",
    "This is amazing",
    "Very good experience",
    "I am happy with the service",
    "Excellent quality",

    "I hate this",
    "Very bad experience",
    "This is terrible",
    "I am disappointed",
    "Worst product ever"
]

labels = [
    1, 1, 1, 1, 1,   # Positive
    0, 0, 0, 0, 0    # Negative
]
print("Number of Sample",len(texts))

Number of Sample 10


step 4 : LOAD BERT Tokenizer
BERT does not directly understand raw text.
The tokenizer processes the raw text by splitting it into tokens, adding special tokens, and converting them into numerical input IDs and attention masks so that the model can understand and process the data.

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
encodings = tokenizer(texts, truncation=True, padding=True,max_length=32,return_tensors='pt')
print(encodings.keys())
print(encodings['input_ids'].shape)
print(encodings['attention_mask'].shape)
#

KeysView({'input_ids': tensor([[ 101, 1045, 2293, 2023, 4031,  102,    0,    0],
        [ 101, 2023, 2003, 6429,  102,    0,    0,    0],
        [ 101, 2200, 2204, 3325,  102,    0,    0,    0],
        [ 101, 1045, 2572, 3407, 2007, 1996, 2326,  102],
        [ 101, 6581, 3737,  102,    0,    0,    0,    0],
        [ 101, 1045, 5223, 2023,  102,    0,    0,    0],
        [ 101, 2200, 2919, 3325,  102,    0,    0,    0],
        [ 101, 2023, 2003, 6659,  102,    0,    0,    0],
        [ 101, 1045, 2572, 9364,  102,    0,    0,    0],
        [ 101, 5409, 4031, 2412,  102,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,

CREATE A CUSTOM DATASET CLASS

---





In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [2]:
from tensorflow.keras.datasets import imdb

vocab_size = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
print("Max review length:", max(len(x) for x in X_train))
print("Min review length:", min(len(x) for x in X_train))

Max review length: 2494
Min review length: 11


In [4]:
word_index = imdb.get_word_index()
index_to_word = {i+3: word for word, i in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
print(" ".join(index_to_word[i] for i in X_train[0]))

<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for wha